In [10]:
penalty_matrix = {
    "A": {"A": 0, "C": 4, "G": 2, "T": 4, "-": 8},
    "C": {"A": 4, "C": 0, "G": 4, "T": 2, "-": 8},
    "G": {"A": 2, "C": 4, "G": 0, "T": 4, "-": 8},
    "T": {"A": 4, "C": 2, "G": 4, "T": 0, "-": 8},
    "-": {"A": 8, "C": 8, "G": 8, "T": 8, "-": 0},
}

for a in penalty_matrix:
    for b in penalty_matrix:
        assert penalty_matrix[a][b] == penalty_matrix[b][a]


def penalty(a, b):
    return penalty_matrix[a][b]

In [12]:
def global_alignment_dp(a, b):
    matrix = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        matrix[i][0] = matrix[i - 1][0] + penalty(a[i - 1], "-")
    for j in range(1, len(b) + 1):
        matrix[0][j] = matrix[0][j - 1] + penalty("-", b[j - 1])
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            als_1 = matrix[i - 1][j - 1] + penalty(a[i - 1], b[j - 1])
            als_2 = matrix[i - 1][j] + penalty(a[i - 1], "-")
            als_3 = matrix[i][j - 1] + penalty("-", b[j - 1])
            matrix[i][j] = min(als_1, als_2, als_3)

    return matrix[len(a)][len(b)]

In [15]:
x = "TACCAGATTCGA"
y = "TACCAGATCGA"
global_alignment_dp(x, y)

8

## Test cases for `global_alignment_dp`

*Generated by Claude.* Each assertion targets a specific part of the algorithm so that, if any test fails, the failure localises the bug:

- **Test 1** — init paths (no inner loop work)
- **Test 2** — identical strings: all-diagonal path with cost 0
- **Test 3** — single substitution: `als_1` (diagonal) with non-zero penalty
- **Test 4** — single indel routed through `als_2` (from above)
- **Test 5** — single indel routed through `als_3` (from left)
- **Test 6** — substitution-vs-indel selection by `min`
- **Test 7** — longer realistic strings

In [17]:
# 1. Init paths only (no inner loop work)
assert global_alignment_dp("", "") == 0
assert global_alignment_dp("AAAA", "") == 32  # first-column init: 4 gaps × 8
assert global_alignment_dp("", "AAAA") == 32  # first-row init:    4 gaps × 8

# 2. Identical strings → all-diagonal path with cost 0
assert global_alignment_dp("ACGT", "ACGT") == 0

# 3. Single substitution (diagonal cell, non-zero penalty lookup)
assert global_alignment_dp("A", "G") == 2  # transition  (purine↔purine)
assert global_alignment_dp("A", "T") == 4  # transversion (purine↔pyrimidine)
assert global_alignment_dp("T", "C") == 2  # transition  (pyrimidine↔pyrimidine)
assert global_alignment_dp("C", "G") == 4  # transversion

# 4. Single indel routed through als_2 (from above)
# Optimal path: A-A match, then a consumes C with gap on b's side, then G-G match.
assert global_alignment_dp("ACG", "AG") == 8

# 5. Single indel routed through als_3 (from left) — mirror of test 4
# Optimal path: A-A match, then b consumes C with gap on a's side, then G-G match.
assert global_alignment_dp("AG", "ACG") == 8

# 6. Substitution should beat a pair of indels
# C→G transition costs 2; two indels would cost 16.
assert global_alignment_dp("ACGT", "AGGT") == 4

# 7. Longer realistic strings
assert global_alignment_dp("TACCAGATTCGA", "TACCAGATTCGA") == 0  # identical
assert global_alignment_dp("TACCAGATTCGA", "TACCAGATTCGT") == 4  # last A↔T transversion
assert global_alignment_dp("TACCAGATTCGA", "TACCAGATTCG") == 8  # one trailing deletion

print("All tests passed.")

All tests passed.
